In [31]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
from StocProcess.RBM import MakeRBMTransProbFunc
from QAE.LowDepthQAE import LowDepthQAE

In [32]:
# RBM parameters
c = -1
d = 1
x0 = 0.5 * (c + d)
t0 = 0
t = 0.6
mu = 0.5
sigma = 1.0
n_terms = 5

# QAE setting
nShot = 12
epsilon = 0.0029
nRep = 10

In [33]:
np.random.seed(1)

In [34]:
# PDF at time t
transProbFunc = MakeRBMTransProbFunc(t, t0, c, d, mu, sigma, n_terms)

# Prob(X > (c + d)/2)
integFunc = lambda x: transProbFunc(x, x0)
pTrue, _ = sp.integrate.quad(integFunc, x0, 1.0)

In [35]:
Ns = (2 ** np.linspace(3, 7, 9)).astype(int)
print(Ns)

[  8  11  16  22  32  45  64  90 128]


In [36]:
retDf = pd.DataFrame(columns=['N', 'pTrue', 'beta', 'pEst', 'absErr', 'totalQueryNum', 'maxDepth'])

for _ in range(nRep):
    for iN in range(len(Ns)):
        N = Ns[iN]
        beta = np.log(N**0.5) / np.log(1 / epsilon)
        qaeRes = LowDepthQAE(pTrue, epsilon, nShot, beta)
        pEst = qaeRes.aEst
        totalQueryNum = qaeRes.TotalQueryNum * N * (N+1) / 2
        maxDepth = qaeRes.MaxDepth * N
        retDf.loc[len(retDf)] = [N, pTrue, beta, pEst, abs(pEst - pTrue), totalQueryNum, maxDepth]

c:\Users\koich\Desktop\Code\DivQCOSDE\QAE\MaximizeL.py:11: RuntimeWarning: divide by zero encountered in log
  neglogL = lambda theta: -np.dot(n1s, np.log(np.sin(thetaMuls * theta)**2)) - np.dot(n0s, np.log(np.cos(thetaMuls * theta)**2))


In [37]:
retDf

,N,pTrue,beta,pEst,absErr,totalQueryNum,maxDepth
0,8.0,0.649605,0.177942,0.650465,0.00086,309312.0,1944.0
1,11.0,0.649605,0.205192,0.649765,0.00016,702504.0,2277.0
2,16.0,0.649605,0.237255,0.649065,0.00054,1873536.0,2768.0
3,22.0,0.649605,0.264506,0.649765,0.00016,4341480.0,3234.0
4,32.0,0.649605,0.296569,0.649365,0.00024,11683584.0,3872.0
...,...,...,...,...,...,...,...
85,32.0,0.649605,0.296569,0.649165,0.00044,11683584.0,3872.0
86,45.0,0.649605,0.325743,0.649865,0.00026,28901340.0,4635.0
87,64.0,0.649605,0.355883,0.649965,0.00036,73432320.0,5568.0
88,90.0,0.649605,0.385057,0.649365,0.00024,180933480.0,6570.0


In [38]:
retDf.groupby('N')['absErr'].mean()

N
8.0      0.000394
11.0     0.000298
16.0     0.000278
22.0     0.000212
32.0     0.000356
45.0     0.000310
64.0     0.000310
90.0     0.000228
128.0    0.000138
Name: absErr, dtype: float64

In [39]:
retDf.groupby('N')['maxDepth'].mean()

N
8.0      1944.0
11.0     2277.0
16.0     2768.0
22.0     3234.0
32.0     3872.0
45.0     4635.0
64.0     5568.0
90.0     6570.0
128.0    7808.0
Name: maxDepth, dtype: float64

In [40]:
retDf.to_csv('RBM_LowDepthQAE.csv', index=False)